___


___

# Conceptos básicos de spacy

**spaCy** (https://spacy.io/) es una biblioteca Python de código abierto que analiza y "comprende" grandes volúmenes de texto. Hay modelos separados disponibles que se adaptan a idiomas específicos (inglés, francés, alemán, etc.).

En esta sección, instalaremos y configuraremos spaCy para que funcione con Python, y luego presentaremos algunos conceptos relacionados con el procesamiento del lenguaje natural.

# Instalación y configuración

La instalación es un proceso de dos pasos. Primero, instale spaCy usando conda o pip. A continuación, descargue el modelo específico que desee, según el idioma.<br> Para obtener más información, visite https://spacy.io/usage/

### 1. Desde la línea de comando o terminal:
> `conda install -c conda-forge spacy`
> <br>*o*<br>
> `pip install -U spacy`

> ### Alternativamente, puede crear un entorno virtual:
> `conda create -n spacyenv python=3 spacy=2`

### 2. A continuación, también desde la línea de comandos (debe ejecutar esto como administrador o usar sudo):

> `python -m spacy download en`

> ### Si tiene éxito, debería ver un mensaje como:

> **`Linking successful`**<br>
> ` C:\Anaconda3\envs\spacyenv\lib\site-packages\en_core_web_sm -->`<br>
> ` C:\Anaconda3\envs\spacyenv\lib\site-packages\spacy\data\en`<br>
> ` `<br>
> ` You can now load the model via spacy.load('en')`


# Trabajar con spaCy en Python

Este es un conjunto típico de instrucciones para importar y trabajar con spaCy. No se sorprenda si esto lleva un tiempo: spaCy tiene una biblioteca bastante grande para cargar:

In [1]:
# Importar spaCy y cargar la biblioteca de idiomas
import spacy
nlp = spacy.load('en_core_web_sm')

# Crear un objeto Doc
doc = nlp(u'Tesla is looking at buying U.S. startup for $6 million')

# Imprime cada ficha por separado
for token in doc:
    print(token.text, token.pos_, token.dep_)

Tesla PROPN nsubj
is VERB aux
looking VERB ROOT
at ADP prep
buying VERB pcomp
U.S. PROPN compound
startup NOUN dobj
for ADP prep
$ SYM quantmod
6 NUM compound
million NUM pobj


Esto no parece muy fácil de usar, pero de inmediato vemos que suceden algunas cosas interesantes:
1. Se reconoce que Tesla es un nombre propio, no solo una palabra al comienzo de una oración.
2. U.S. se mantiene unido como una sola entidad (lo llamamos "token")

A medida que profundicemos en spaCy, veremos qué significa cada una de estas abreviaturas y cómo se derivan. También veremos cómo spaCy puede interpretar los últimos tres tokens combinados `$6 millones` como una referencia a ***dinero***.

___
# Objetos spaCy

Después de importar el módulo espacial en la celda de arriba, cargamos un **modelo** y lo llamamos `nlp`.<br>Luego, creamos un objeto **Doc** aplicando el modelo a nuestro texto y lo llamamos `doc `.<br>spaCy también crea un objeto complementario **Vocab** que cubriremos en secciones posteriores.<br>El objeto **Doc** que contiene el texto procesado es nuestro enfoque aquí.

___
# Pipeline
Cuando ejecutamos `nlp`, nuestro texto ingresa a una *tubería de procesamiento* que primero desglosa el texto y luego realiza una serie de operaciones para etiquetar, analizar y describir los datos. Fuente de la imagen: https://spacy.io/usage/spacy-101#pipelines

<img src="pipeline.png" width="600">

Podemos verificar qué componentes se encuentran actualmente en la tubería. En secciones posteriores, aprenderemos cómo deshabilitar componentes y agregar otros nuevos según sea necesario.

In [2]:
nlp.pipeline

[('tagger', <spacy.pipeline.Tagger at 0x7f8a199e2d90>),
 ('parser', <spacy.pipeline.DependencyParser at 0x7f8a1b062dd0>),
 ('ner', <spacy.pipeline.EntityRecognizer at 0x7f8a1b0803b0>)]

In [3]:
nlp.pipe_names

['tagger', 'parser', 'ner']

___
## Tokenización
El primer paso en el procesamiento de texto es dividir todas las partes componentes (palabras y puntuación) en "tokens". Estos tokens se anotan dentro del objeto Doc para contener información descriptiva. Entraremos en muchos más detalles sobre la tokenización en el siguiente notebook. Por ahora, veamos un ejemplo:

In [4]:
doc2 = nlp(u"Tesla isn't   looking into startups anymore.")

for token in doc2:
    print(token.text, token.pos_, token.dep_)

Tesla PROPN nsubj
is VERB aux
n't ADV neg
   SPACE 
looking VERB ROOT
into ADP prep
startups NOUN pobj
anymore ADV advmod
. PUNCT punct


Observe cómo `isn't` se ha dividido en dos tokens. spaCy reconoce tanto el verbo raíz `is` como la negación adjunta. Observe también que tanto el espacio en blanco extendido como el punto al final de la oración tienen asignados sus propios tokens.

Es importante tener en cuenta que aunque `doc2` contiene información procesada sobre cada token, también conserva el texto original:

In [5]:
doc2

Tesla isn't   looking into startups anymore.

In [6]:
doc2[0]

Tesla

In [7]:
type(doc2)

spacy.tokens.doc.Doc

___
## Etiquetado de parte del discurso - Part-of-Speech Tagging (POS)
El siguiente paso después de dividir el texto en tokens es asignar partes del discurso. En el ejemplo anterior, `Tesla` fue reconocido como un ***nombre propio***. Aquí se requiere algún modelo estadístico. Por ejemplo, las palabras que siguen a "the" suelen ser sustantivos.

Para obtener una lista completa de etiquetas POS, visite https://spacy.io/api/annotation#pos-tagging

In [8]:
doc2[0].pos_

'PROPN'

___
## Dependencias
También analizamos las dependencias sintácticas asignadas a cada token. `Tesla` se identifica como `nsubj` o el ***sujeto nominal*** de la oración.

Para obtener una lista completa de dependencias sintácticas, visite https://spacy.io/api/annotation#dependency-parsing
<br>Puede encontrar una buena explicación de las dependencias escritas [aquí](https://nlp.stanford.edu/software/dependencies_manual.pdf)

In [9]:
doc2[0].dep_

'nsubj'

Para ver el nombre completo de una etiqueta, use `spacy.explain(tag)`

In [10]:
spacy.explain('PROPN')

'proper noun'

In [11]:
spacy.explain('nsubj')

'nominal subject'

___
## Atributos de token adicionales
Los veremos de nuevo en próximos notebooks. Por ahora solo queremos ilustrar parte de la otra información que spaCy asigna a los tokens:

|Tag|Descripción|doc2[0].tag|
|:------|:------:|:------|
|`.text`|The original word text<!-- .element: style="text-align:left;" -->|`Tesla`|
|`.lemma_`|The base form of the word|`tesla`|
|`.pos_`|The simple part-of-speech tag|`PROPN`/`proper noun`|
|`.tag_`|The detailed part-of-speech tag|`NNP`/`noun, proper singular`|
|`.shape_`|The word shape – capitalization, punctuation, digits|`Xxxxx`|
|`.is_alpha`|Is the token an alpha character?|`True`|
|`.is_stop`|Is the token part of a stop list, i.e. the most common words of the language?|`False`|

In [12]:
# Lemas (la forma base de la palabra):
print(doc2[4].text)
print(doc2[4].lemma_)

looking
look


In [13]:
# Partes del habla simples y etiquetas detalladas:
print(doc2[4].pos_)
print(doc2[4].tag_ + ' / ' + spacy.explain(doc2[4].tag_))

VERB
VBG / verb, gerund or present participle


In [14]:
# Formas de palabras:
print(doc2[0].text+': '+doc2[0].shape_)
print(doc[5].text+' : '+doc[5].shape_)

Tesla: Xxxxx
U.S. : X.X.


In [15]:
# Valores booleanos:
print(doc2[0].is_alpha)
print(doc2[0].is_stop)

True
False


___
## Spans
A veces puede ser difícil trabajar con objetos grandes de Doc. Un **span** (duración) es una porción del objeto Doc en la forma `Doc[start:stop]`.

In [16]:
doc3 = nlp(u'Although commmonly attributed to John Lennon from his song "Beautiful Boy", \
the phrase "Life is what happens to us while we are making other plans" was written by \
cartoonist Allen Saunders and published in Reader\'s Digest in 1957, when Lennon was 17.')

In [17]:
life_quote = doc3[16:30]
print(life_quote)

"Life is what happens to us while we are making other plans"


In [18]:
type(life_quote)

spacy.tokens.span.Span

En próximos notebooks veremos cómo crear objetos Span usando `Span()`. Esto nos permitirá asignar información adicional al Span.

___
## Oraciones (sentences)
Ciertos tokens dentro de un objeto Doc también pueden recibir una etiqueta de "comienzo de oración". Si bien esto no crea inmediatamente una lista de oraciones, estas etiquetas permiten la generación de segmentos de oraciones a través de `Doc.sents`. Más tarde escribiremos nuestras propias reglas de segmentación.

In [19]:
doc4 = nlp(u'This is the first sentence. This is another sentence. This is the last sentence.')

In [20]:
for sent in doc4.sents:
    print(sent)

This is the first sentence.
This is another sentence.
This is the last sentence.


In [21]:
doc4[6].is_sent_start

True

## A continuación: Tokenización